# MESSIDOR × MAPLES-DR — Data Matching & Integration
**Objective:** จับคู่ภาพต้นฉบับ (MESSIDOR) กับ Segmentation Masks (MAPLES-DR)  
สร้าง Verified Catalog ที่พร้อมใช้กับ PyTorch DataLoader

---
## Pipeline Overview
```
MESSIDOR (raw .tif/.jpg)
    └── index file paths  (Step 1)
MAPLES-DR (train/test masks)
    └── scan mask paths   (Step 1)
           │
           ▼
    ID-level matching     (Step 1)
           │
           ▼
    Physical existence    (Step 2)
    verification
           │
           ▼
    Final verified pivot  (Step 2)
           │
           ▼
    Copy to MAPLES_Matched (Step 3)
```

## 0. Configuration & Imports

0a. Lib import

In [2]:
import os  # Path management
import glob # Searching .tif/jpg and Mask .png
import shutil # File management
from pathlib import Path #Path management

import pandas as pd # use for Indexing, Matching, Filter Data & Pivot table
from tqdm import tqdm #Showing progress bar

0b. Path configuration 

In [3]:
base_dir = Path.cwd()

# input path
messidor_dir = os.path.join(base_dir, "MESSIDOR")
maples_dr_dir = os.path.join(base_dir, "MAPLES-DR")
collection_dir = os.path.join(base_dir, "MAPLES_Matched")

# Derived output paths
out_images = os.path.join(collection_dir, "images")
out_masks = os.path.join(collection_dir, "masks")

splits = ['train', 'test']

---
## Step 1 — Index & ID-Level Matching
Create `Image_ID → file path` mapping for MESSIDOR  
and scan Mask every class from MAPLES-DR and macting them by Image_ID

1. Index MESSIDOR images

In [4]:
messidor_files = glob.glob(os.path.join(messidor_dir, "**","*.tif"), recursive = True)
if not messidor_files:
    messidor_files = glob.glob(os.path.join(messidor_dir, "**", "*.jpg"),recursive= True)

- Map:  stem (image with no extension) → full path

In [5]:
messidor_map = {os.path.splitext(os.path.basename(f))[0]: f for f in messidor_files}
print(f'MESSIDOR images indexed: {len(messidor_map):,} ')

MESSIDOR images indexed: 1,200 


2. Scan MAPPLES-DR masks and match to MESSIDOR

In [6]:
matches =[]
for split in splits:
    split_path = os.path.join(maples_dr_dir, split)
    if not os.path.isdir(split_path):
        print(f"[Warning] Split folder not found: {split_path}")
        continue

    classes = [d for d in os.listdir(split_path)
               if os.path.isdir(os.path.join(split_path, d))]

    for cls in classes:
        cls_path = os.path.join(split_path, cls)
        for m_path in glob.glob(os.path.join(cls_path,"*.png")):
            mask_id = os.path.splitext(os.path.basename(m_path))[0]
            if mask_id in messidor_map:
                matches.append({
                    "Image_ID": mask_id,
                    'Original_Image_Path': messidor_map[mask_id],
                    'Mask_Path': m_path,
                    'Class': cls,
                    'Split': split
                })
df_matches = pd.DataFrame(matches)

- Total Image and Class pairing

In [7]:
print(f'Total matching (image, class) pairs: {len(df_matches):,}')
print(f'Unique Image IDs matched: {df_matches['Image_ID'].nunique():,}')

Total matching (image, class) pairs: 2,369
Unique Image IDs matched: 198


- Total Classes found

In [8]:
class_summary =(
    df_matches["Class"]
    .value_counts()
    .sort_index()
    .reset_index()
)
class_summary.columns = ['Class', 'Count']

print('Classes found')
class_summary


Classes found


,Class,Count
0,BrightUncertains,198
1,CottonWoolSpots,198
2,Drusens,198
3,Exudates,198
4,Hemorrhages,198
5,Macula,197
6,Microaneurysms,198
7,Neovascularization,198
8,OpticCup,192
9,OpticDisc,198


- Sample of Mapped Images

In [9]:
display(df_matches.head())

,Image_ID,Original_Image_Path,Mask_Path,Class,Split
0,20051019_38557_0100_PP,d:\Backup\Senior-Project\Pre-Train\MESSIDOR\Ba...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,BrightUncertains,train
1,20051020_55346_0100_PP,d:\Backup\Senior-Project\Pre-Train\MESSIDOR\Ba...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,BrightUncertains,train
2,20051020_55701_0100_PP,d:\Backup\Senior-Project\Pre-Train\MESSIDOR\Ba...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,BrightUncertains,train
3,20051020_58065_0100_PP,d:\Backup\Senior-Project\Pre-Train\MESSIDOR\Ba...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,BrightUncertains,train
4,20051020_58214_0100_PP,d:\Backup\Senior-Project\Pre-Train\MESSIDOR\Ba...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,BrightUncertains,train


---
## Step 2 — Physical Existence Verification
Check whether the Image and Mask files are on the disk.
→ Missing files will be flagged to prevent DataLoader from crashing.

1.Verification

In [10]:
verified_rows = []
missing_log =[]

In [11]:
for _,row in tqdm(df_matches.iterrows(), total=len(df_matches), desc="Verifying files"):
    img_ok = os.path.exists(row['Original_Image_Path'])
    mask_ok = os.path.exists(row['Mask_Path'])

    if img_ok and mask_ok:
        verified_rows.append(row)
    else:
        missing_log.append({
            "Image_ID": row['Image_ID'],
            "Image_Missing": not img_ok,
            'Mask_Missing': not mask_ok
        })
df_verified = pd.DataFrame(verified_rows)

Verifying files: 100%|██████████| 2369/2369 [00:01<00:00, 1817.40it/s]


- Verification Summary

In [12]:
print(f'Verified pairs: {len(df_verified):,}')
print(f'Missing entries: {len(missing_log)}')

if missing_log:
    print("\n[!] Missing files:")
    display(pd.DataFrame(missing_log))

Verified pairs: 2,369
Missing entries: 0


2. Pivot to final Catalog (1 row/image)
- Matching All mask to 1 Fundus image

In [16]:
final_cat = (
    df_verified.pivot_table(
        index = ['Image_ID','Original_Image_Path','Split'],
        columns='Class',
        values='Mask_Path',
        aggfunc='first'
    ).reset_index()
)
final_cat.columns.name = None
print(f'Unigue verified images: {len(final_cat):,}')
display(final_cat.head())

Unigue verified images: 198


,Image_ID,Original_Image_Path,Split,BrightUncertains,CottonWoolSpots,Drusens,Exudates,Hemorrhages,Macula,Microaneurysms,Neovascularization,OpticCup,OpticDisc,RedUncertains,Vessels
0,20051019_38557_0100_PP,d:\Backup\Senior-Project\Pre-Train\MESSIDOR\Ba...,train,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...
1,20051020_44923_0100_PP,d:\Backup\Senior-Project\Pre-Train\MESSIDOR\Ba...,test,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...
2,20051020_55346_0100_PP,d:\Backup\Senior-Project\Pre-Train\MESSIDOR\Ba...,train,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,NaN,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...
3,20051020_55701_0100_PP,d:\Backup\Senior-Project\Pre-Train\MESSIDOR\Ba...,train,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...
4,20051020_57566_0100_PP,d:\Backup\Senior-Project\Pre-Train\MESSIDOR\Ba...,test,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...,d:\Backup\Senior-Project\Pre-Train\MAPLES-DR\t...


---
## Step 3 — Data Integration
Coppy Original Image & Mask that verrified to 'MAPLES_Matched'
Folder Structure:
```
MAPLES_Matched/
├── images/         ← original fundus images
└── masks/
    ├── OpticDisc/
    ├── Macula/
    ├── Exudates/
    └── ...         ← one subfolder per class
```

1. Create output directory

In [17]:
if os.path.exists(collection_dir):
    shutil.rmtree(collection_dir)
os.makedirs(out_images, exist_ok=True)
print(f'Output directory ready: {collection_dir}')

Output directory ready: d:\Backup\Senior-Project\Pre-Train\MAPLES_Matched


2. Copy Images & Masks

In [21]:
for _, row in tqdm(final_cat.iterrows(), total = len(final_cat), desc='Integrating'):
    img_id = row['Image_ID']
    img_src = row['Original_Image_Path']

    # Original images
    img_ext = os.path.splitext(img_src)[1]
    shutil.copy2(img_src, os.path.join(out_images, f'{img_id}{img_ext}'))
    # Verified mask for this image
    img_mask = df_verified[df_verified['Image_ID']==img_id]
    for _, mask_row in img_mask.iterrows():
        cls_dir = os.path.join(out_masks, mask_row['Class'])
        os.makedirs(cls_dir, exist_ok=True)
        shutil.copy2(mask_row["Mask_Path"], os.path.join(cls_dir, f"{img_id}.png"))
n_images  = len(os.listdir(out_images))
n_classes = len(os.listdir(out_masks))
print("\n=== Integration Complete ===")
print(f"  Images  : {n_images}")
print(f"  Classes : {n_classes}")
print(f"  Output  : {collection_dir}")

Integrating: 100%|██████████| 198/198 [00:06<00:00, 29.46it/s]


=== Integration Complete ===
  Images  : 198
  Classes : 12
  Output  : d:\Backup\Senior-Project\Pre-Train\MAPLES_Matched
